# Optotagging analysis from NWB

This notebook is the NWB-based rewrite of Anna's optotagging pipeline
(`optotagging_analysis.py` + `plotting_funcs.py` + `main.py`). Spikes, waveforms,
peak channels and QC are read from an AIND ephys NWB file loaded with `NWBUtils`.
Laser onsets and stimulation parameters are read from the **raw** Open Ephys asset,
exactly as Anna does.

| Original source                       | Source used here |
|---------------------------------------|----------------------|
| NIDAQ event stream (laser onsets)     | raw NIDAQ events folder (channel 2), same as Anna |
| `*opto.csv` (trial parameters)        | raw `*opto.csv`, same as Anna |
| SpikeInterface `sorting_output`       | `nwb_data.units['spike_times'][u]` |
| waveform extractor templates          | `nwb_data.units['waveform_mean'][u]` |
| `extremum_channels`                   | `nwb_data.units['extremum_channel_index']` |
| probe / stream name                   | `nwb_data.units['device_name']` |
| `default_qc` / `decoder_label`        | `get_units_passed_default_qc(nwb_data)` |

Laser **onset times** are read from the raw Open Ephys NIDAQ digital-input events
(channel 2, label `'2'`) and the **stimulation parameters** from the raw `*opto.csv`.
Only spikes, waveforms, peak channel and QC come from the NWB. The raw
`ecephys_clipped` asset must be attached to the capsule alongside the sorted NWB.


## Setup

In [1]:
import sys
from pathlib import Path

%load_ext autoreload
%autoreload 2

MODULE_PATH = Path("/root/capsule/src/aind_dft_ephys_analysis")
if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))

import numpy as np
import pandas as pd

from nwb_utils import NWBUtils
from optotagging_Anna_nwb import (
    OptotaggingAnalysisNWB,
    find_recording_clipped_folder,
    read_opto_trials_csv,
    get_laser_onsets_from_nidaq,
)
import optotagging_Anna_nwb_plotting as opto_plot

print(f"✅ Modules loaded from: {MODULE_PATH}")


✅ Modules loaded from: /root/capsule/src/aind_dft_ephys_analysis


## Step 1 — Load the ephys NWB

In [2]:
SESSION_NAME = "ecephys_839480_2026-06-02_16-20-58_sorted_2026-06-09_15-56-14"
SAVE_FOLDER = "/root/capsule/results"

nwb_data = NWBUtils.read_ephys_nwb(session_name=SESSION_NAME)
assert nwb_data is not None, "Failed to load ephys NWB — check the session name."
print("Session:", getattr(nwb_data, "session_id", SESSION_NAME))
print("Probes (device_name):")
from optotagging_Anna_nwb import get_stream_names
print(" ", get_stream_names(nwb_data))


Found ephys NWB: /root/capsule/data/ecephys_839480_2026-06-02_16-20-58_sorted_2026-06-09_15-56-14/nwb/ecephys_839480_2026-06-02_16-20-58_experiment1_recording1.nwb
Successfully read ephys NWB from: /root/capsule/data/ecephys_839480_2026-06-02_16-20-58_sorted_2026-06-09_15-56-14/nwb/ecephys_839480_2026-06-02_16-20-58_experiment1_recording1.nwb
Session: ecephys_839480_2026-06-02_16-20-58
Probes (device_name):
  ['Probe B', 'Probe C']


## Step 2 — Locate the raw asset (NIDAQ events + opto CSV)

The laser onsets and parameters come from the raw Open Ephys `ecephys_clipped`
folder. This cell finds it, previews the `*opto.csv` parameters, and reads the
NIDAQ channel-2 onsets. If auto-detection fails, set `RECORDING_CLIPPED_FOLDER`
(and optionally `TRIALS_CSV`) explicitly.


In [3]:
# Set these explicitly if auto-detection fails; otherwise leave as None.
RECORDING_CLIPPED_FOLDER = None
TRIALS_CSV = None
LASER_EVENT_ID = "2"   # NIDAQ channel-2 digital-input label
OPTO_RECORDING = 0     # segment index with the laser stimulation
FLIP_NIDAQ = False     # subtract 0.5 s if the sync signal was flipped

clipped = RECORDING_CLIPPED_FOLDER or find_recording_clipped_folder(SESSION_NAME)
print("ecephys_clipped folder:", clipped)

opto_csv = read_opto_trials_csv(clipped, TRIALS_CSV)
print(f"opto CSV rows: {len(opto_csv)}; columns: {list(opto_csv.columns)}")

onsets = get_laser_onsets_from_nidaq(
    clipped, event_id=LASER_EVENT_ID, opto_recording=OPTO_RECORDING, flip_NIDAQ=FLIP_NIDAQ
)
print(f"NIDAQ onsets: {len(onsets)} (should match CSV rows)")
opto_csv.head()


ecephys_clipped folder: /root/capsule/data/ecephys_839480_2026-06-02_16-20-58/ecephys/ecephys_clipped
opto CSV rows: 420; columns: ['site', 'power', 'param_group', 'emission_location', 'duration', 'rise_time', 'num_pulses', 'pulse_interval', 'wavelength', 'type', 'interval']
NIDAQ onsets: 420 (should match CSV rows)


,site,power,param_group,emission_location,duration,rise_time,num_pulses,pulse_interval,wavelength,type,interval
0,0,2.0,train,Probe B,10,1,5,40,638,external_red,1.16
1,0,2.0,train,Probe B,10,1,5,40,473,external_blue,0.97
2,0,2.0,train,Probe B,10,1,5,40,638,external_red,0.90
3,0,2.0,train,Probe B,10,1,5,40,473,external_blue,0.94
4,0,0.5,train,Probe B,10,1,5,40,638,external_red,0.87


## Step 3 — Build the analyzer

In [4]:
analysis = OptotaggingAnalysisNWB(
    nwb_data=nwb_data,
    session_name=SESSION_NAME,
    recording_clipped_folder=clipped,
    trials_csv=TRIALS_CSV,
    laser_event_id=LASER_EVENT_ID,
    opto_recording=OPTO_RECORDING,
    flip_NIDAQ=FLIP_NIDAQ,
)

print(f"QC-passing units: {len(analysis.qc_units)}")
print(f"Laser onsets: {len(analysis.laser_onset_times)}")

# trial types present (falls back to a single 'all' type if no 'type' column)
if "type" in analysis.trial_ids.columns:
    trial_types = list(np.unique(analysis.trial_ids["type"]))
else:
    trial_types = ["all"]
print("Trial types:", trial_types)


Number of units passing QC: 274
QC-passing units: 274
Laser onsets: 420
Trial types: ['external_blue', 'external_red']


## Step 4 — Compute laser-response metrics per probe and save CSVs

In [5]:
# Query defines which parameter combinations get analyzed, mirroring main.py.
# Adjust keys to the columns your stimulus table actually has.
powers = list(np.unique(analysis.trial_ids["power"])) if "power" in analysis.trial_ids.columns else [None]
trials_query = {
    "type": trial_types,
    "power": powers,
}
suffixes = [None, "mW"]  # one per trials_query key; matches column naming in main.py

all_metrics = {}
for probe in analysis.get_stream_names():
    metrics = analysis.one_probe_laser_responses(
        trials_query=trials_query,
        probe=probe,
        suffixes=suffixes,
        ignore_onset_offset=True,
        pre_opto_duration=None,  # set to a float (s) to compute pre-stim ISI / rate
    )
    if len(metrics) == 0:
        continue
    metrics = OptotaggingAnalysisNWB.add_best_power_columns(metrics, trial_types)
    all_metrics[probe] = metrics

    Path(SAVE_FOLDER).mkdir(parents=True, exist_ok=True)
    out_csv = Path(SAVE_FOLDER) / f"{analysis.session}_{probe}_laser_response_metrics.csv"
    metrics.to_csv(out_csv, index=False)
    print(f"Saved {out_csv} ({len(metrics)} units)")

/root/capsule/src/aind_dft_ephys_analysis/optotagging_Anna_nwb.py:756: RuntimeWarning: Mean of empty slice
  metrics.at[row_i, f"{col}_mean_time_to_first_spike"] = np.nanmean(ttfs)
/root/capsule/src/aind_dft_ephys_analysis/optotagging_Anna_nwb.py:757: RuntimeWarning: Mean of empty slice
  metrics.at[row_i, f"{col}_mean_jitter"] = np.nanmean(jitter)
/root/capsule/src/aind_dft_ephys_analysis/optotagging_Anna_nwb.py:756: RuntimeWarning: Mean of empty slice
  metrics.at[row_i, f"{col}_mean_time_to_first_spike"] = np.nanmean(ttfs)
/root/capsule/src/aind_dft_ephys_analysis/optotagging_Anna_nwb.py:757: RuntimeWarning: Mean of empty slice
  metrics.at[row_i, f"{col}_mean_jitter"] = np.nanmean(jitter)
/root/capsule/src/aind_dft_ephys_analysis/optotagging_Anna_nwb.py:756: RuntimeWarning: Mean of empty slice
  metrics.at[row_i, f"{col}_mean_time_to_first_spike"] = np.nanmean(ttfs)
/root/capsule/src/aind_dft_ephys_analysis/optotagging_Anna_nwb.py:757: RuntimeWarning: Mean of empty slice
  metrics.

Saved /root/capsule/results/ecephys_839480_2026-06-02_16-20-58_sorted_2026-06-09_15-56-14_Probe B_laser_response_metrics.csv (16 units)
Saved /root/capsule/results/ecephys_839480_2026-06-02_16-20-58_sorted_2026-06-09_15-56-14_Probe C_laser_response_metrics.csv (258 units)


## Step 5 — Select tagged units and make plots

Selection criteria mirror `main.py` (adjust the thresholds / trial-type names to
match your data).

In [7]:
def tagged_units(metrics, trial_type, min_sig_pulses=4, max_jitter=0.01, max_isi=0.5):
    q = []
    if f"{trial_type}_train_max_num_sig_pulses" in metrics.columns:
        q.append(f"{trial_type}_train_max_num_sig_pulses >= {min_sig_pulses}")
    if f"{trial_type}_train_best_mean_jitter" in metrics.columns:
        q.append(f"{trial_type}_train_best_mean_jitter < {max_jitter}")
    if "pre_stim_isi_ratio" in metrics.columns:
        q.append(f"pre_stim_isi_ratio < {max_isi}")
    if not q:
        return metrics.iloc[0:0]
    return metrics.query(" and ".join(q))


for probe, metrics in all_metrics.items():
    for trial_type in trial_types:
        tagged = tagged_units(metrics, trial_type)
        unit_ids = tagged["unit_id"].astype(int).tolist()
        print(f"{probe} / {trial_type}: {len(unit_ids)} tagged units -> {unit_ids}")
        if not unit_ids:
            continue
        base = f"{analysis.session}_{probe}_{trial_type}_responsive"
        opto_plot.multi_unit_raster_plot(
            analysis, unit_ids, trial_types, probe, base, save_folder=SAVE_FOLDER
        )
        opto_plot.multi_unit_pulse_plot(
            analysis, unit_ids, metrics, trial_types, probe, base + "_pulse_plot",
            save_folder=SAVE_FOLDER,
        )

Probe B / external_blue: 6 tagged units -> [24, 91, 98, 99, 100, 101]


/root/capsule/src/aind_dft_ephys_analysis/optotagging_Anna_nwb_plotting.py:120: UserWarning: constrained_layout not applied because axes sizes collapsed to zero.  Try making figure larger or Axes decorations smaller.
  fig.savefig(out, dpi=150)


/root/capsule/results/ecephys_839480_2026-06-02_16-20-58_sorted_2026-06-09_15-56-14_Probe B_external_blue_responsive.png saved
/root/capsule/results/ecephys_839480_2026-06-02_16-20-58_sorted_2026-06-09_15-56-14_Probe B_external_blue_responsive_pulse_plot.png saved
Probe B / external_red: 1 tagged units -> [31]
/root/capsule/results/ecephys_839480_2026-06-02_16-20-58_sorted_2026-06-09_15-56-14_Probe B_external_red_responsive.png saved
/root/capsule/results/ecephys_839480_2026-06-02_16-20-58_sorted_2026-06-09_15-56-14_Probe B_external_red_responsive_pulse_plot.png saved
Probe C / external_blue: 0 tagged units -> []
Probe C / external_red: 0 tagged units -> []


In [ ]:
# Close the NWB IO handle when done
if hasattr(nwb_data, "io"):
    nwb_data.io.close()